In [ ]:
"""
dataset.py

Creates PyTorch Dataset and DataLoader objects for the U-Net PINN.

Expected preprocessed arrays:

    X:
        (N, 10, 256, 256)

    Y:
        (N, 1, 256, 256)

    K:
        (N, 1, 256, 256)

Each dataset item returns:

    inputs
    target_lst
    conductivity
"""

import numpy as np
import torch

from torch.utils.data import Dataset
from torch.utils.data import DataLoader


class LSTDataset(Dataset):
    """
    PyTorch Dataset for the U-Net PINN.
    """

    def __init__(
        self,
        inputs,
        targets,
        conductivity
    ):
        self.inputs = inputs
        self.targets = targets
        self.conductivity = conductivity

        # Make sure all datasets have the same
        # number of samples.

        if not (
            len(self.inputs)
            == len(self.targets)
            == len(self.conductivity)
        ):
            raise ValueError(
                "Inputs, targets, and conductivity "
                "must contain the same number of samples."
            )

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):

        inputs = torch.tensor(
            self.inputs[index],
            dtype=torch.float32
        )

        target_lst = torch.tensor(
            self.targets[index],
            dtype=torch.float32
        )

        conductivity = torch.tensor(
            self.conductivity[index],
            dtype=torch.float32
        )

        return (
            inputs,
            target_lst,
            conductivity
        )


def create_dataloaders(
    X,
    Y,
    K,
    batch_size=2,
    train_ratio=0.70,
    validation_ratio=0.15
):
    """
    Split the data into training, validation, and
    testing sets and create DataLoaders.

    Parameters
    ----------
    X : numpy.ndarray
        U-Net inputs.
        Shape: (N, 10, 256, 256)

    Y : numpy.ndarray
        Target LST.
        Shape: (N, 1, 256, 256)

    K : numpy.ndarray
        Conductivity.
        Shape: (N, 1, 256, 256)

    batch_size : int
        Number of samples processed at once.

    train_ratio : float
        Fraction of data used for training.

    validation_ratio : float
        Fraction of data used for validation.

    Returns
    -------
    train_loader
    validation_loader
    test_loader
    """

    # --------------------------------------------------------
    # Check sample counts
    # --------------------------------------------------------

    if not (
        len(X)
        == len(Y)
        == len(K)
    ):
        raise ValueError(
            "X, Y, and K must contain the same "
            "number of samples."
        )

    # --------------------------------------------------------
    # Check input shapes
    # --------------------------------------------------------

    if X.ndim != 4:
        raise ValueError(
            f"X must be 4-dimensional, got {X.ndim}D."
        )

    if Y.ndim != 4:
        raise ValueError(
            f"Y must be 4-dimensional, got {Y.ndim}D."
        )

    if K.ndim != 4:
        raise ValueError(
            f"K must be 4-dimensional, got {K.ndim}D."
        )

    if X.shape[1] != 10:
        raise ValueError(
            f"X must contain 10 input channels, "
            f"got {X.shape[1]}."
        )

    if Y.shape[1] != 1:
        raise ValueError(
            "Y must contain exactly 1 channel."
        )

    if K.shape[1] != 1:
        raise ValueError(
            "K must contain exactly 1 channel."
        )

    # --------------------------------------------------------
    # Create reproducible random split
    # --------------------------------------------------------

    total_samples = len(X)

    rng = np.random.default_rng(
        seed=42
    )

    indices = rng.permutation(
        total_samples
    )

    train_size = int(
        total_samples * train_ratio
    )

    validation_size = int(
        total_samples * validation_ratio
    )

    train_indices = indices[
        :train_size
    ]

    validation_indices = indices[
        train_size:
        train_size + validation_size
    ]

    test_indices = indices[
        train_size + validation_size:
    ]

    # --------------------------------------------------------
    # Split input data
    # --------------------------------------------------------

    X_train = X[
        train_indices
    ]

    X_validation = X[
        validation_indices
    ]

    X_test = X[
        test_indices
    ]

    # --------------------------------------------------------
    # Split target data
    # --------------------------------------------------------

    Y_train = Y[
        train_indices
    ]

    Y_validation = Y[
        validation_indices
    ]

    Y_test = Y[
        test_indices
    ]

    # --------------------------------------------------------
    # Split conductivity data
    # --------------------------------------------------------

    K_train = K[
        train_indices
    ]

    K_validation = K[
        validation_indices
    ]

    K_test = K[
        test_indices
    ]

    # --------------------------------------------------------
    # Create Dataset objects
    # --------------------------------------------------------

    train_dataset = LSTDataset(
        X_train,
        Y_train,
        K_train
    )

    validation_dataset = LSTDataset(
        X_validation,
        Y_validation,
        K_validation
    )

    test_dataset = LSTDataset(
        X_test,
        Y_test,
        K_test
    )

    # --------------------------------------------------------
    # Create DataLoaders
    # --------------------------------------------------------

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    # --------------------------------------------------------
    # Display split information
    # --------------------------------------------------------

    print(
        "Training samples:",
        len(train_dataset)
    )

    print(
        "Validation samples:",
        len(validation_dataset)
    )

    print(
        "Testing samples:",
        len(test_dataset)
    )

    return (
        train_loader,
        validation_loader,
        test_loader
    )


if __name__ == "__main__":

    # --------------------------------------------------------
    # Test the Dataset and DataLoader using fake data.
    # --------------------------------------------------------

    X = np.random.rand(
        10,
        10,
        256,
        256
    ).astype(
        np.float32
    )

    Y = np.random.rand(
        10,
        1,
        256,
        256
    ).astype(
        np.float32
    )

    K = np.random.rand(
        10,
        1,
        256,
        256
    ).astype(
        np.float32
    )

    (
        train_loader,
        validation_loader,
        test_loader
    ) = create_dataloaders(
        X,
        Y,
        K,
        batch_size=2
    )

    # Get one batch.

    inputs, target_lst, conductivity = next(
        iter(train_loader)
    )

    print(
        "\nBatch shapes:"
    )

    print(
        "Inputs:",
        inputs.shape
    )

    print(
        "Target LST:",
        target_lst.shape
    )

    print(
        "Conductivity:",
        conductivity.shape
    )